# 01 — Data Collection

Pull the raw project data from the Urban Institute [Education Data Portal](https://educationdata.urban.org/) API. We collect three school-level datasets for 2021:

- **CRDC AP/IB enrollment** — Advanced Placement participation (our outcome of interest).
- **CCD directory** — school characteristics (location, type, enrollment, staffing, lunch program).
- **CCD enrollment by grade** — grade 9–12 counts, later used as the denominator for the AP participation rate.

Each dataset is downloaded page by page from the paginated API and written to `../data/` as a raw CSV. Cleaning and merging happen in notebook `02`.

In [1]:
# Import packages
import pandas as pd
import requests

## 1. Explore the API with a small test

Before downloading nationwide, we query a single state, Rhode Island (`fips = 44`) to confirm each endpoint responds and to inspect the response format. These test calls also show that race and sex are broken out by code, and that `race = 99`, `sex = 99` return the all-students totals we want.

In [2]:
# use Rhode Island('fips': 44) school to test CCD data availability
# 'fips' source: https://www.census.gov/library/reference/code-lists/ansi/ansi-codes-for-states.html#accordion-455dd4b1cd-item-2c5946f1b7
# headers needed here, make request as google chrome, not python
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'}

url = 'https://educationdata.urban.org/api/v1/schools/ccd/directory/2021/'
response = requests.get(url, params={'fips': 44}, headers=headers)
print(response.status_code)
print(response.text[:500])

200
{"count":319,"next":null,"previous":null,"results":[{"year":2021,"ncessch":"440000100380","school_id":"4400380","school_name":"R.I. School for the Deaf","leaid":"4400001","lea_name":"R.I. Sch for the Deaf","state_leaid":"RI-50","seasch":"50-28702","street_mailing":"One Corliss Park","city_mailing":"Providence","state_mailing":"RI","zip_mailing":"02908","street_location":"One Corliss Park","city_location":"Providence","state_location":"RI","zip_location":"02908","phone":"(401)222-3525","fips":44,


In [3]:
# use Rhode Island('fips': 44) school to test CRDC AP data availability
crdc_url = 'https://educationdata.urban.org/api/v1/schools/crdc/ap-ib-enrollment/2021/race/sex/'

response = requests.get(crdc_url, params={'fips': 44}, headers=headers) 
print(response.status_code)

if response.status_code == 200:
    crdc_json = response.json()
    print('total count:', crdc_json['count'])
    crdc_test = pd.DataFrame(crdc_json['results'])
    print(crdc_test.shape)
    print(crdc_test.columns.tolist())
    print(crdc_test.head(10))

200
total count: 7392
(7392, 17)
['crdc_id', 'year', 'ncessch', 'fips', 'leaid', 'race', 'sex', 'disability', 'lep', 'enrl_IB', 'enrl_AP', 'enrl_AP_science', 'enrl_AP_math', 'enrl_AP_other', 'enrl_AP_language', 'enrl_gifted_talented', 'enrl_AP_compsci']
        crdc_id  year       ncessch  fips    leaid  race  sex  disability  \
0  440000100380  2021  440000100380    44  4400001     1    1          99   
1  440000100380  2021  440000100380    44  4400001     2    1          99   
2  440000100380  2021  440000100380    44  4400001     3    1          99   
3  440000100380  2021  440000100380    44  4400001     4    1          99   
4  440000100380  2021  440000100380    44  4400001     5    1          99   
5  440000100380  2021  440000100380    44  4400001     6    1          99   
6  440000100380  2021  440000100380    44  4400001     7    1          99   
7  440000100380  2021  440000100380    44  4400001    99    1          99   
8  440000100380  2021  440000100380    44  4400001   

In [4]:
# keep only the total rows: race and sex = 99 (all)
# https://educationdata.urban.org/documentation/schools.html

response = requests.get(crdc_url, params={'fips': 44, 'race': 99, 'sex': 99}, headers=headers)
print(response.status_code)

if response.status_code == 200:
    crdc_json = response.json()
    print('total count:', crdc_json['count'])
    crdc_test = pd.DataFrame(crdc_json['results'])
    print(crdc_test.shape)
    print(crdc_test[['ncessch', 'race', 'sex', 'enrl_AP']].head(10))

200
total count: 308
(308, 17)
        ncessch  race  sex  enrl_AP
0  440000100380    99   99       -2
1  440000300121    99   99        8
2  440000400379    99   99      394
3  440000500271    99   99       -2
4  440000600326    99   99       -2
5  440000800297    99   99       30
6  440000800493    99   99       -2
7  440000900413    99   99       -2
8  440001400472    99   99       -2
9  440001500473    99   99       -2


## 2. Download the full CRDC AP/IB enrollment (2021)

Loop through every page of the paginated endpoint (all-students totals only) and save the combined result to `../data/crdc_ap_2021_raw.csv`.

In [5]:
# get CRDC AP data
url = 'https://educationdata.urban.org/api/v1/schools/crdc/ap-ib-enrollment/2021/race/sex/?race=99&sex=99'
crdc_results = []

# keep going until no next page
while url is not None:
    response = requests.get(url, headers=headers)
    data_json = response.json()
    crdc_results.extend(data_json['results'])   
    print(len(crdc_results), 'of', data_json['count'])
    url = data_json['next']

crdc = pd.DataFrame(crdc_results)
print(crdc.shape)
crdc.to_csv('../data/crdc_ap_2021_raw.csv', index=False)

10000 of 98010
20000 of 98010
30000 of 98010
40000 of 98010
50000 of 98010
60000 of 98010
70000 of 98010
80000 of 98010
90000 of 98010
98010 of 98010
(98010, 17)


## 3. Download the full CCD school directory (2021)

Same pagination pattern, saved to `../data/ccd_directory_2021_raw.csv`.

In [6]:
# get CCD data
url = 'https://educationdata.urban.org/api/v1/schools/ccd/directory/2021/'
ccd_results = []

while url is not None:
    response = requests.get(url, headers=headers)
    data_json = response.json()
    ccd_results.extend(data_json['results'])
    print(len(ccd_results), 'of', data_json['count'])
    url = data_json['next']

ccd = pd.DataFrame(ccd_results)
print(ccd.shape)
ccd.to_csv('../data/ccd_directory_2021_raw.csv', index=False)

10000 of 102130
20000 of 102130
30000 of 102130
40000 of 102130
50000 of 102130
60000 of 102130
70000 of 102130
80000 of 102130
90000 of 102130
100000 of 102130
102130 of 102130
(102130, 52)


### Sanity check: do CRDC school IDs appear in CCD?

Confirm the two files can be joined on the school ID (`ncessch`) before we rely on it in notebook `02`.

In [7]:
# read the raw data, read ID-like columns as text to depress warning
crdc = pd.read_csv('../data/crdc_ap_2021_raw.csv', dtype={'ncessch': str, 'crdc_id': str, 'leaid': str})
ccd = pd.read_csv('../data/ccd_directory_2021_raw.csv', dtype={'ncessch': str, 'leaid': str,
                         'state_leg_district_lower': str, 'state_leg_district_upper': str})

# drop CRDC rows without a school ID
crdc_with_id = crdc[crdc['ncessch'].notna()]

# True if the CRDC school ID also appears in CCD, False otherwise
matched = crdc_with_id['ncessch'].isin(ccd['ncessch'])

print('CRDC schools with an ID:', len(crdc_with_id))
print('found in CCD:', matched.sum())
# match rate
print('match rate:', matched.mean())

CRDC schools with an ID: 97179
found in CCD: 97015
match rate: 0.998312392595108


## 4. Download CCD enrollment by grade 9–12 (2021)

We keep K–12 and high schools (those offering grade 12) and need their grade 9–12 enrollment as the AP-rate denominator. First a single-state/single-grade test to inspect the endpoint, then the full pull looping over grades 9–12. Saved to `../data/ccd_enrollment_grade9to12_2021_raw.csv`.

In [8]:
# CCD enrollment by grade, 2021, test with Rhode Island (fips = 44) and grade 9
url = 'https://educationdata.urban.org/api/v1/schools/ccd/enrollment/2021/grade-9/?fips=44'
response = requests.get(url, headers=headers)
print(response.status_code)

if response.status_code == 200:
    data_json = response.json()
    print('total count:', data_json['count'])
    test = pd.DataFrame(data_json['results'])
    print(test.columns.tolist())
    print(test.head(10))

200
total count: 69
['year', 'ncessch', 'ncessch_num', 'grade', 'race', 'sex', 'enrollment', 'fips', 'leaid']
   year       ncessch   ncessch_num  grade  race  sex  enrollment  fips  \
0  2021  440000100380  440000100380      9    99   99         6.0    44   
1  2021  440000300121  440000300121      9    99   99       203.0    44   
2  2021  440000400379  440000400379      9    99   99       255.0    44   
3  2021  440000800297  440000800297      9    99   99        44.0    44   
4  2021  440000900413  440000900413      9    99   99         7.0    44   
5  2021  440001500488  440001500488      9    99   99       125.0    44   
6  2021  440001700477  440001700477      9    99   99        46.0    44   
7  2021  440001800478  440001800478      9    99   99        42.0    44   
8  2021  440001900479  440001900479      9    99   99       148.0    44   
9  2021  440002100538  440002100538      9    99   99        80.0    44   

     leaid  
0  4400001  
1  4400003  
2  4400004  
3  4400008  

In [9]:
# In order to keep K-12 schools(to see the comparison), pulling CCD enrollment for grades 9-12 to use as the denominator
grade_results = []

for grade in [9, 10, 11, 12]:
    url = f'https://educationdata.urban.org/api/v1/schools/ccd/enrollment/2021/grade-{grade}/'
    while url is not None:
        response = requests.get(url, headers=headers)
        data_json = response.json()
        grade_results.extend(data_json['results'])
        print('grade', grade, ':', len(grade_results), 'rows so far')
        url = data_json['next']

ccd_grades = pd.DataFrame(grade_results)
print(ccd_grades.shape)
ccd_grades.to_csv('../data/ccd_enrollment_grade9to12_2021_raw.csv', index=False)

grade 9 : 10000 rows so far
grade 9 : 20000 rows so far
grade 9 : 28030 rows so far
grade 10 : 38030 rows so far
grade 10 : 48030 rows so far
grade 10 : 55868 rows so far
grade 11 : 65868 rows so far
grade 11 : 75868 rows so far
grade 11 : 83729 rows so far
grade 12 : 93729 rows so far
grade 12 : 103729 rows so far
grade 12 : 111511 rows so far
(111511, 9)


### Sanity check: do grade-12 schools have grade 9–12 enrollment?

In [10]:
# match rate: how many schools in our sample have grade 9-12 enrollment
# only schools have 12 grade count
ccd_grade12 = ccd[ccd['highest_grade_offered'] == 12]

matched = ccd_grade12['ncessch'].isin(ccd_grades['ncessch'])
print('schools offering grade 12:', len(ccd_grade12))
print('found in the grade 9-12 data:', matched.sum())
print('match rate:', matched.mean())

schools offering grade 12: 27645
found in the grade 9-12 data: 27621
match rate: 0.9991318502441671
